<a href="https://colab.research.google.com/github/20233l001039-coder/Curso_Llama/blob/main/Hands_On_Fundamentos_de_LLMs_con_Modelos_Llama.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **HANDS - ON: FUNDAMENTOS DE LLMS CON MODELOS LLAMA**


Una vez vista la masterclass ***Fundamentos de LLMs y Arquitectura de Llama***, se proporciona el siguiente ***Colab*** para construir, en vivo, la primera llamada a Llama y medir su comportamiento.

Usamos **Groq** para tener acceso a inferencia de Llama con GPU gratuita, sin necesidad de infraestructura propia.

De igual manera, se proporciona la **solución** de este notebook a través del siguiente [enlace](https://colab.research.google.com/drive/1bjMHw7fmMxo9dyQp140e9mqotKRkfKBg?usp=sharing).

## **CONFIGURACIÓN DEL ENTORNO**

Antes de llamar a Llama necesitamos activar la GPU gratuita de Colab y conectar nuestra cuenta de **Groq**. La librería `groq` es el cliente oficial en Python para hacer llamadas a la API.

### **COLAB SECRETS**

Para no exponer tu ***API key*** directamente en el código, Colab ofrece un panel de ***Secrets*** (ícono de llave en la barra lateral izquierda) donde puedes guardarla de forma segura, añadiendo un nombre asociado a la ***API key*** para guardarla dentro de una variable y usarla dentro del notebook.

In [ ]:
# Instalar cliente de Groq y leer API key desde Colab Secrets
!pip install groq -q

import os
from groq import Groq
from google.colab import userdata

client = Groq(api_key=userdata.get('GROQ_API_KEY'))
print("Cliente de Groq inicializado correctamente.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 4.6 MB/s eta 0:00:00
Cliente de Groq inicializado correctamente.


## **DE PALABRAS A TOKENS**

Antes de que Llama genere una sola palabra, tu prompt se divide en **tokens**. Vamos a definir un prompt de ejemplo que después enviaremos a Llama.

In [ ]:
# Definir un prompt de ejemplo (una pregunta real de tu propio contexto)
prompt = "¿Cuál es la diferencia entre la RAM y el almacenamiento en una computadora?"
# prompt = "Explica en un párrafo qué hace un router doméstico"

print(prompt)

¿Cuál es la diferencia entre la RAM y el almacenamiento en una computadora?


### **PRIMERA LLAMADA A LLAMA (ZERO-SHOT)**

Un LLM predice el siguiente token más probable, **no busca la respuesta en una base de datos**. Vamos a enviar el prompt a Llama en modo zero-shot (sin ejemplos previos) y ver la respuesta generada.

In [ ]:
# Enviar el prompt a Llama en Groq y mostrar la respuesta generada
response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt}]
)

print(response.choices[0].message.content)

**RAM (Memoria de Acceso Aleatorio) vs. Almacenamiento (Disco duro, SSD, etc.)**

| Característica | RAM | Almacenamiento (HDD/SSD) |
|----------------|-----|--------------------------|
| **Propósito principal** | Guarda datos e instrucciones que el procesador necesita usar de inmediato. | Guarda datos a largo plazo: el sistema operativo, programas, archivos personales, etc. |
| **Volatilidad** | Volátil: pierde su contenido cuando la computadora se apaga o reinicia. | Permanente: conserva la información aunque el dispositivo se apague. |
| **Velocidad** | Extremadamente rápida (nanosegundos). | Más lenta (microsegundos para SSD, milisegundos para HDD). |
| **Capacidad típica** | Pocas GB (4 GB, 8 GB, 16 GB, 32 GB). | Desde 128 GB hasta varios terabytes (1 TB, 2 TB, 4 TB, etc.). |
| **Costo por gigabyte** | Alto: es más caro por GB que el almacenamiento. | Bajo: el costo por GB es mucho menor. |
| **Acceso a datos** | Acceso aleatorio y simultáneo a cualquier posición. | Acceso aleator

### **CUÁNTOS TOKENS CONSUMIÓ EL PROMPT**

La respuesta de la API trae un resumen (`usage`) con el número de tokens de entrada y de salida. Es la forma más directa de responder: ¿cuántos tokens consumió mi prompt?

In [ ]:
# Mostrar cuántos tokens tuvo el prompt y cuántos tuvo la respuesta
print("Tokens del prompt:", response.usage.prompt_tokens)
print("Tokens de la respuesta:", response.usage.completion_tokens)
print("Tokens totales:", response.usage.total_tokens)
# response.usage.total_tokens es lo que se factura por esta llamada

Tokens del prompt: 86
Tokens de la respuesta: 723
Tokens totales: 809


### **MIDIENDO LA LATENCIA DE INFERENCIA**

Elegir el tamaño correcto de modelo es una decisión de ingeniería, no solo de potencia bruta. Vamos a medir cuánto tarda Llama en responder el mismo prompt.

In [ ]:
# Medir el tiempo de respuesta de Llama para el mismo prompt
import time

inicio = time.time()
response_tiempo = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt}]
)
duracion = time.time() - inicio
# duracion_ms = duracion * 1000  # si prefieres reportarlo en milisegundos

print(f"Tiempo de respuesta: {duracion:.2f} segundos")

Tiempo de respuesta: 1.14 segundos


### **COMPARANDO DOS TAMAÑOS DE MODELO**

Repite la misma llamada usando la versión más grande de Llama disponible en Groq y compara tiempo de respuesta y calidad contra el modelo ligero.

In [ ]:
# Repetir la llamada con un modelo Llama más grande y comparar tiempo y calidad
inicio = time.time()
response_grande = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[{"role": "user", "content": prompt}]
)
duracion_grande = time.time() - inicio

print(f"Modelo ligero: {duracion:.2f} s — {response.usage.total_tokens} tokens")
print(f"Modelo grande: {duracion_grande:.2f} s — {response_grande.usage.total_tokens} tokens")
print("\nRespuesta del modelo grande:\n", response_grande.choices[0].message.content)

Modelo ligero: 1.14 s — 809 tokens
Modelo grande: 2.16 s — 949 tokens

Respuesta del modelo grande:
 **RAM vs. Almacenamiento: cuál es la diferencia y por qué importa**

| Característica | **RAM (Memoria de Acceso Aleatorio)** | **Almacenamiento (disco duro, SSD, etc.)** |
|----------------|----------------------------------------|--------------------------------------------|
| **Función principal** | Mantener los datos y programas que la CPU está usando **en ese momento**. | Guardar de forma permanente todo lo que quieres conservar: sistema operativo, aplicaciones, documentos, fotos, etc. |
| **Volatilidad** | **Volátil**: se pierde cuando apagas o reinicias la máquina. | **No volátil**: los datos permanecen aunque la computadora esté apagada. |
| **Velocidad** | Muy rápida (nanosegundos). Permite a la CPU leer/escribir datos al instante. | Más lenta (micro‑ a milisegundos). Los SSD son mucho más rápidos que los HDD, pero aún quedan detrás de la RAM. |
| **Capacidad típica** | De 4 GB

## **CHALLENGE: COMPARADOR DE MODELOS LLAMA**

Una vez visto el ***Hands-On: Fundamentos de LLMs***, se presenta el siguiente reto para que el alumnado pueda repasar y reforzar lo aprendido dentro de la clase.

Se construirá un pequeño **comparador que envíe 3 preguntas** reales a Llama y registre, para cada una, el **tiempo de respuesta** y el **número de tokens** consumidos.

**IMPORTANTE:** Para su revisión, **es indispensable que los apartados anteriores se encuentren llenados con el código visto durante la sesión.**

### **INSTRUCCIONES:**

**1. Carga la API key y genera 3 preguntas:**

   * Lee la API key de llama ya configurada desde **Colab Secrets**.
   * Construye una lista vacía llamada `preguntas` y agrégale 3 preguntas frecuentes de tu propio contexto (soporte técnico, tienda, escuela, etc.).

In [ ]:
# Leer API key desde Colab Secrets
from google.colab import userdata
from groq import Groq

api_key = userdata.get('GROQ_API_KEY')
client = Groq(api_key=api_key)

In [ ]:
# Definir la lista de preguntas
preguntas = []

preguntas.append("¿Cómo puedo restablecer mi contraseña si olvidé mi correo de recuperación?")
preguntas.append("¿Cuál es el tiempo estimado de respuesta para un ticket de prioridad alta?")
preguntas.append("¿Cómo activo la autenticación de dos factores en mi cuenta?")

**2. Consulta la primera pregunta:** Envía la primera pregunta a Llama usando el modelo más ligero disponible en Groq. Guarda la respuesta, el tiempo de respuesta y el número de tokens en un diccionario llamado `resultado_1`.

In [ ]:
# Consultar la primera pregunta y guardar el resultado en resultado_1
import time

inicio = time.time()

response_1 = client.chat.completions.create(
    model="openai/gpt-oss-20b",  # Modelo ligero actual en Groq
    messages=[{"role": "user", "content": preguntas[0]}]
)

tiempo_1 = time.time() - inicio

resultado_1 = {
    "pregunta": preguntas[0],
    "respuesta": response_1.choices[0].message.content,
    "tiempo_respuesta": round(tiempo_1, 2),
    "tokens_consumidos": response_1.usage.total_tokens
}

**3. Repite para la segunda y tercera pregunta:** Crea `resultado_2` y `resultado_3` de la misma forma.

In [ ]:
# Consultar la segunda pregunta y guardar el resultado en resultado_2
inicio = time.time()

response_2 = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": preguntas[1]}]
)

tiempo_2 = time.time() - inicio

resultado_2 = {
    "pregunta": preguntas[1],
    "respuesta": response_2.choices[0].message.content,
    "tiempo_respuesta": round(tiempo_2, 2),
    "tokens_consumidos": response_2.usage.total_tokens
}

In [ ]:
# Consultar la tercera pregunta y guardar el resultado en resultado_3
inicio = time.time()

response_3 = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": preguntas[2]}]
)

tiempo_3 = time.time() - inicio

resultado_3 = {
    "pregunta": preguntas[2],
    "respuesta": response_3.choices[0].message.content,
    "tiempo_respuesta": round(tiempo_3, 2),
    "tokens_consumidos": response_3.usage.total_tokens
}

**4. Junta los resultados:** Agrega `resultado_1`, `resultado_2` y `resultado_3` a una lista vacía llamada `resultados`.

In [ ]:
# Definir la lista resultados y agregar los tres diccionarios
resultados = []
resultados.append(resultado_1)
resultados.append(resultado_2)
resultados.append(resultado_3)

**5. Muestra la comparación:** Imprime `resultados` y concluye si el modelo ligero resolvió las 3 preguntas satisfactoriamente.

In [ ]:
# Mostrar la tabla final de resultados
from tabulate import tabulate

# Construir la tabla con los datos más relevantes
tabla = []
for i, res in enumerate(resultados, 1):
    tabla.append([
        i,
        res["pregunta"][:50] + "...",
        res["respuesta"][:80] + "...",
        f"{res['tiempo_respuesta']} seg",
        res["tokens_consumidos"]
    ])

# Imprimir la tabla con encabezados
print(tabulate(
    tabla,
    headers=["#", "Pregunta", "Respuesta (inicio)", "Tiempo", "Tokens"],
    tablefmt="grid"
))

# Conclusión
print("\n=== CONCLUSIÓN ===")
print(f"Tiempo total: {sum(r['tiempo_respuesta'] for r in resultados):.2f} segundos")
print(f"Tokens totales: {sum(r['tokens_consumidos'] for r in resultados)}")
print("El modelo ligero resolvió las 3 preguntas satisfactoriamente, con respuestas coherentes y tiempos bajos.")

+-----+-------------------------------------------------------+-------------------------------------------------------------------------------------+----------+----------+
|   # | Pregunta                                              | Respuesta (inicio)                                                                  | Tiempo   |   Tokens |
+=====+=======================================================+=====================================================================================+==========+==========+
|   1 | ¿Cómo puedo restablecer mi contraseña si olvidé mi... | ¡Claro! Si has olvidado tu correo de recuperación pero necesitas restablecer tu ... | 1.86 seg |     1537 |
+-----+-------------------------------------------------------+-------------------------------------------------------------------------------------+----------+----------+
|   2 | ¿Cuál es el tiempo estimado de respuesta para un t... | En la mayoría de las organizaciones con políticas de SLA (Service‑Level Agre